### Web Scraping

In [ ]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import plotly.express as px
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output

#### Extrayendo las ofertas de empleo (Data Science) en Linkedin del último mes

In [6]:
option = webdriver.ChromeOptions()
option.add_argument("--window-size=1300,700")

driver = webdriver.Chrome(options=option)
    
try:
    driver.get("https://www.linkedin.com")  
    driver.find_element(By.XPATH, '//a[@href="https://www.linkedin.com/jobs/search?trk=guest_homepage-basic_guest_nav_menu_jobs"]').click()
    
    input_workstation = driver.find_element(By.XPATH, '//input[@aria-controls="job-search-bar-keywords-typeahead-list"]')
    input_workstation.send_keys("Data Science")
    input_location = driver.find_element(By.XPATH, '//input[@aria-controls="job-search-bar-location-typeahead-list"]')
    input_location.clear()
    input_location.send_keys("Argentina")
    input_location.send_keys(Keys.RETURN)   
    
    driver.find_element(By.XPATH, '//button[@aria-label="Filtro «Fecha de publicación». Se ha aplicado el filtro «Cualquier momento». Al hacer clic en este botón, se muestran todas las opciones del filtro «Fecha de publicación»."]').click() 
    driver.find_element(By.ID, 'f_TPR-1').click()  
    driver.find_element(By.CLASS_NAME, 'filter__submit-button').click()
    
    workstation = driver.find_elements(By.CLASS_NAME, 'base-search-card__title')
    company = driver.find_elements(By.CLASS_NAME, 'hidden-nested-link')
    location = driver.find_elements(By.CLASS_NAME, 'job-search-card__location')
    time = driver.find_elements(By.CLASS_NAME, 'job-search-card__listdate')
    
except Exception as e:
    print(f"Ha ocurrido el siguiente error: {e}")
    driver.quit()
    
# -------- Una vez obtenidos los datos los organizamos en un Data Frame --------

data = {
    "workstation": [],
    "company": [],
    "location": [],
    "time": []
}

len_col = np.array([len(workstation),len(location),len(company),len(time)])
min_len_col = len_col.min()

for w in workstation:
    data['workstation'].append(w.text)
    if len(data['workstation']) == min_len_col:
        break
for c in company:
    data['company'].append(c.text)
    if len(data['company']) == min_len_col:
        break
for l in location:
    data['location'].append(l.text)
    if len(data['location']) == min_len_col:
        break
for t in time:
    data['time'].append(t.text)
    if len(data['time']) == min_len_col:
        break

df = pd.DataFrame(data)
df

,workstation,company,location,time
0,Analista de Datos Comerciales Jr.,Rappi,Buenos Aires y alrededores,Hace 6 días
1,Analista de Datos Comerciales Jr.,Rappi,"Provincia de Buenos Aires, Argentina",Hace 3 semanas
2,Product Analyst,Naranja X,Argentina,Hace 2 días
3,Analista de datos,B. Braun Group,"Provincia de Buenos Aires, Argentina",Hace 3 semanas
4,Data Scientist AR,Premier Media,Argentina,Hace 3 días
5,Data Science Analyst Jr,Accenture Argentina,Buenos Aires y alrededores,Hace 2 días
6,Inteligencia Artificial Data Analyst,"Esselección ""Gestionando Talento Humano""","San Miguel de Tucumán, Tucumán, Argentina",Hace 2 días
7,Data Scientist,InvGate,Argentina,Hace 2 semanas
8,Analista de Modelo de riesgo crediticio - Fintech,Mercado Libre,"Provincia de Buenos Aires, Argentina",Hace 1 semana
9,Data Scientist Sr,LA NACION,"Vicente López, Provincia de Buenos Aires, Arge...",Hace 1 semana


### Data Wrangling

In [7]:
print(df["location"].value_counts())

# Mapeo de datos: dado que muchas ubicaciones son similares y sobreinforman, las reemplazamos y resumimos la iformación

data_mapping = {
    "Ciudad Autónoma de Buenos Aires, Provincia de Buenos Aires, Argentina": "Ciudad Autónoma de Buenos Aires, Argentina",
    "Buenos Aires, Provincia de Buenos Aires, Argentina": "Provincia de Buenos Aires, Argentina",
    "Buenos Aires y alrededores": "Provincia de Buenos Aires, Argentina"
}

df["location"] = df["location"].apply(lambda x : data_mapping.get(x, x))
print("---------------------------------------------------------------------------")
print(df["location"].value_counts())

location
Argentina                                                                18
Buenos Aires, Provincia de Buenos Aires, Argentina                       11
Buenos Aires y alrededores                                               10
Provincia de Buenos Aires, Argentina                                      4
Vicente López, Provincia de Buenos Aires, Argentina                       2
Ciudad Autónoma de Buenos Aires, Provincia de Buenos Aires, Argentina     2
San Miguel de Tucumán, Tucumán, Argentina                                 1
Dock Sud, Provincia de Buenos Aires, Argentina                            1
Mendoza, Mendoza, Argentina                                               1
Name: count, dtype: int64
---------------------------------------------------------------------------
location
Provincia de Buenos Aires, Argentina                   25
Argentina                                              18
Vicente López, Provincia de Buenos Aires, Argentina     2
Ciudad Autónoma de Bue

In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(id="body",className="e8_body",children=[
    html.H1("Ofertas de trabajo en Linkedin", className="e8_title"),
        dcc.Dropdown(id="dropdown",className="e8_dropdown",
                    options = [
                        {"label":"Ubiación (País, Ciudad, Barrio)","value":"location"},
                        {"label":"Tiempo de publicación","value":"time"},
                    ],
                    value="location",
                    multi=False,
                    clearable=False),
    html.Div(className="e8_div",children=[
        dcc.Graph(id="graph_1",className="e8_graph",figure={}),
        dcc.Graph(id="graph_2",className="e8_graph",figure={})
    ])
])

@app.callback(
    [Output(component_id="graph_1",component_property="figure"),
    Output(component_id="graph_2",component_property="figure")],
    [Input(component_id="dropdown",component_property="value")]
)

def update_graph(slct_var):
    values_count = df[slct_var].value_counts().reset_index()
    barplot = px.bar(values_count, x=slct_var, y="count", title='Cantidades contadas')
    if slct_var == "location":
        barplot.update_layout(xaxis=dict(tickfont=dict(size=7)))
    piechart = px.pie(values_count, values='count', names=slct_var, title='Porcentages de cantidades')
    if slct_var == "location":
        piechart.update_layout(legend=dict(font=dict(size=6)))
    
    return barplot, piechart
    
if __name__ == "__main__":
    app.run_server(debug=False)

##### Como podemos observar lamentablemente gran parte de las ofertas no especifican su ubicación